In [1]:
# imports
import json

import logging

# Set up logging configuration at the top of your notebook or script
logging.basicConfig(
    level=logging.INFO,  # Change to DEBUG for more verbosity
    format='%(asctime)s - %(levelname)s - %(message)s'
)

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-Coder-32B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
   model_name,
   dtype="auto",
   device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

`torch_dtype` is deprecated! Use `dtype` instead!
2025-10-12 07:30:57,896 - INFO - We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/14 [00:00<?, ?it/s]

In [ ]:
def generate_for_prompt(model, tokenizer, user_prompt, **kwargs):
    def build_messages(user_prompt):
        return [
            {
                "role": "system",
                "content": (
                    "You are an expert assistant specialized in generating accurate and high-quality Java code from natural language descriptions. "
                    "When responding, output only the Java code, with no explanations or comments. "
                    "Do not modify or reformat any Java code already provided in the prompt—only continue filling in the code and return the final completed result."
                )
            },
            {"role": "user", "content": user_prompt}
        ]

    max_new_tokens = kwargs.get("max_new_tokens", 1024)
    top_p = kwargs.get("top_p", 0.95)
    temperature = kwargs.get("temperature", 0.1)
    top_k = kwargs.get("top_k", 0)

    messages = build_messages(user_prompt)
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=max_new_tokens,
        # top_p=top_p,
        # temperature=temperature,
        # top_k=top_k
    )

    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    response = tokenizer.batch_decode(
        generated_ids, skip_special_tokens=True)[0]
    return response

In [4]:
def generate_for_prompts(model, tokenizer, prompts, **kwargs):
    completions = []
    for idx, prompt in enumerate(prompts):
        print(f"Generating completion for problem {idx+1}/{len(prompts)}")
        response = generate_for_prompt(model, tokenizer, prompt, **kwargs)
        completions.append(response if isinstance(
            response, list) else [response])

        print(f'Completion for problem {idx+1}:\n{response}\n{"-"*40}\n')

    return completions

In [5]:
def generate_for_dataset(model, tokenizer, problems, **kwargs):
    user_prompts = [problem['prompt'] for problem in problems]
    
    completions = generate_for_prompts(model, tokenizer, user_prompts, **kwargs)
    
    # Save completions to a JSON file
    output_path = 'completions.json'
    with open(output_path, 'w') as f:
        json.dump(completions, f, indent=4)

    return completions


In [6]:
test_prompt = """
import java.util.*;
import java.lang.*;

class Solution {
    /**
        Given a positive floating point number, it can be decomposed into
        and integer part (largest integer smaller than given number) and decimals
        (leftover part always smaller than 1).

        Return the decimal part of the number.
        >>> truncateNumber(3.5)
        0.5
     */
    public double truncateNumber(double number) {        
""";

In [10]:
print(generate_for_prompt(model, tokenizer, test_prompt))

To complete the `truncateNumber` method, you need to calculate the decimal part of a given floating-point number. This can be done by subtracting the integer part of the number from the number itself. In Java, you can use the `Math.floor()` method to get the largest integer less than or equal to the given number. Here's how you can implement this:

```java
import java.util.*;
import java.lang.*;

class Solution {
    /**
        Given a positive floating point number, it can be decomposed into
        an integer part (largest integer smaller than given number) and decimals
        (leftover part always smaller than 1).

        Return the decimal part of the number.
        >>> truncateNumber(3.5)
        0.5
     */
    public double truncateNumber(double number) {
        // Get the integer part of the number
        double integerPart = Math.floor(number);
        // Subtract the integer part from the original number to get the decimal part
        double decimalPart = number - inte

In [8]:
# ds_json_path = 'benchmark/datasets/humaneval-x/humanevalx-java-refined.json'

# problems = json.load(open(ds_json_path))
# logging.info(f'Loaded {len(problems)} problems from "{ds_json_path}"')

# generate_for_dataset(model, tokenizer, ds_json_path)